# MachineGuard AI — Exploratory Data Analysis

This notebook reads the **active** training data source. By default the included data are an independent simulator (NOT the UCI benchmark); if you download the original AI4I benchmark, those records become the active source. Both are synthetic. Run from the repository root.

In [ ]:
from src.data import load_training_data, source_label
from src.constants import FEATURES, TARGET
import pandas as pd
import matplotlib.pyplot as plt
df, audit = load_training_data()
print(source_label(audit["dataset_kind"]))
print(audit)
df.head()

## Check data quality, class imbalance and numeric summary
These checks are not a proof of industrial usefulness.

In [ ]:
display(df.isna().sum().to_frame("missing"))
print("Duplicates (inputs + label):", df.duplicated().sum())
print("Target distribution:")
display(df[TARGET].value_counts(normalize=False).to_frame("records"))
display(df[FEATURES[1:]].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df[TARGET].map({0:"Normal label",1:"Failure label"}).value_counts().plot.bar(ax=axes[0], title="Class imbalance")
df["Rotational speed [rpm]"].hist(ax=axes[1], bins=35)
axes[1].set_title("Spindle RPM distribution")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df.loc[df[TARGET]==0, "Rotational speed [rpm]"],df.loc[df[TARGET]==0, "Torque [Nm]"],s=7,alpha=.2,label="Normal")
ax.scatter(df.loc[df[TARGET]==1, "Rotational speed [rpm]"],df.loc[df[TARGET]==1, "Torque [Nm]"],s=11,alpha=.7,label="Failure")
ax.set(xlabel="RPM",ylabel="Torque (Nm)",title="Machine operating regions (simulated)")
ax.legend(); plt.show()

## Interpretation

The labels are strongly imbalanced; a high accuracy alone may conceal missed failures. Avoid training on `UDI`, `Product ID`, `TWF`, `HDF`, `PWF`, `OSF`, or `RNF`. This repository implements leak-safe preprocessing and training in `src/modeling.py`. Do not infer causal industrial failure mechanisms from these plots.